# PolarisT atlas-unprofiled driver discovery demo

This notebook demonstrates how to define a desired CD8+ T-cell state and use the pretrained PolarisT model to rank atlas-unprofiled candidate genes.

Complete [package installation](../README.md#1-package-installation) before running this notebook. Then follow the steps below.


## Data preparation

Download [Anndata_cd8_raw.h5ad](https://figshare.com/ndownloader/files/66953270) and save it in a local data directory. The dataset is documented in the associated [Figshare record](https://doi.org/10.6084/m9.figshare.32934569).

Set `resource_dir` below to your actual data directory. For example, if the file is saved as `/path/to/polarist_data/Anndata_cd8_raw.h5ad`, use `/path/to/polarist_data`.

**Note:** `resource_dir` must point to the directory containing the file, not to the `.h5ad` file itself.


In [ ]:
# Replace with the directory containing Anndata_cd8_raw.h5ad.
resource_dir = "/path/to/polarist_data"


In [1]:
from polarist import rank_unseen_drivers

## Input parameters

- `phenotype_name`: phenotype label used in output filenames.
- `positive_genes`: genes expected to be highly expressed in the desired state.
- `negative_genes`: genes expected to be weakly expressed in the desired state; optional.
- `extreme_fraction`: fraction selected from each expression-score tail within every dataset. Valid range: `(0, 0.5]`. Default: `0.05`.
- `n_genes`: number of top-ranked atlas-unprofiled genes returned. Accepted values: `100`, `1000`, or `None` for all candidates. Default: `1000`.
- `tf_only`: whether to additionally generate the transcription-factor subset in `result.tf_ranking`.
- `resource_dir`: local directory containing the downloaded `Anndata_cd8_raw.h5ad` file.


## Define the desired phenotype

This example defines a CD8+ T-cell objective with increased stemness and reduced exhaustion. Stemness-associated genes form the positive signature, while exhaustion-associated genes form the negative signature.

To define a custom phenotype, replace `positive_genes` and `negative_genes` and update `phenotype_name`. Use the same `resource_dir` set above. `negative_genes` can be omitted when using a positive signature alone.


In [ ]:
phenotype_name = "stemness"
extreme_fraction = 0.05
n_genes = 1000

positive_genes = [
    "TCF7", "LEF1", "SLAMF6", "SELL", "BCL2",
    "BCL6", "CXCR5", "CCNE1", "CCNE2",
]
negative_genes = ["TOX", "HAVCR2", "ENTPD1", "CD101", "CD244"]


## Rank atlas-unprofiled genes


In [2]:
result = rank_unseen_drivers(
    positive_genes=positive_genes,
    negative_genes=negative_genes,
    phenotype_name=phenotype_name,
    extreme_fraction=extreme_fraction,
    n_genes=n_genes,
    tf_only=True,
    resource_dir=resource_dir,
)

/home/wangyiheng/anaconda3/envs/polarist_0.2.1/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


[1/4] Pretrained model loaded.
[2/4] Defining phenotype from gene signatures.
[3/4] Perturbation-phenotype alignment and ranking completed.
[4/4] Transcription-factor ranking completed (1000 TFs).


## Full ranking

Top atlas-unprofiled candidate genes ranked toward the desired phenotype.

In [3]:
result.full_ranking.head(20)

,rank,gene,score,raw_score,hub_penalty,is_transcription_factor
0,1,KDM2A,3.717777,3.717777,0.0,True
1,2,ZNF217,3.711962,3.711962,0.0,True
2,3,MPPED2,3.611234,3.611234,0.0,False
3,4,PMAIP1,3.597432,3.597432,0.0,False
4,5,TLE4,3.587789,3.587789,0.0,False
5,6,IRF5,3.584649,3.584649,0.0,True
6,7,THBS1,3.567058,3.567058,0.0,False
7,8,EHMT2,3.537706,3.537706,0.0,False
8,9,HDAC7,3.522969,3.522969,0.0,False
9,10,HLA-B,3.516301,3.516301,0.0,False


## Transcription-factor ranking

In [4]:
result.tf_ranking.head(20)

,tf_rank,rank,gene,score,raw_score,hub_penalty,is_transcription_factor
0,1,1,KDM2A,3.717777,3.717777,0.000000,True
1,2,2,ZNF217,3.711962,3.711962,0.000000,True
2,3,6,IRF5,3.584649,3.584649,0.000000,True
3,4,12,PRDM1,3.475162,3.475162,0.000000,True
4,5,13,STAT2,3.469269,3.469269,0.000000,True
5,6,15,IRF9,3.462531,3.462531,0.000000,True
6,7,16,BATF,3.437633,3.437633,0.000000,True
7,8,26,TCF20,3.231426,3.231426,0.000000,True
8,9,27,BCL11A,3.199150,3.199150,0.000000,True
9,10,30,IRF1,3.175505,3.320959,0.072727,True


## Save the rankings

The output filenames are generated automatically from `phenotype_name`.

In [5]:
result.full_ranking.to_csv(
    f"atlas-unprofiled_{result.phenotype_name}_full_ranking.csv",
    index_label="Perturbation",
)
result.tf_ranking.to_csv(
    f"atlas-unprofiled_{result.phenotype_name}_tf_ranking.csv",
    index_label="Perturbation",
)